In [1]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl
import pandas as pd
import json
import sys
import os
from glob import glob
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['font.serif'] = 'Times New Roman'
mpl.rcParams['font.size'] = 9

In [2]:
sys.path

['',
 '/opt/easybuild/lib/python3.9/site-packages',
 '/home/fr/fr_ze12',
 '/home/fr/fr_ze12/.conda/envs/ymaze/lib/python310.zip',
 '/home/fr/fr_ze12/.conda/envs/ymaze/lib/python3.10',
 '/home/fr/fr_ze12/.conda/envs/ymaze/lib/python3.10/lib-dynload',
 '/home/fr/fr_ze12/.conda/envs/ymaze/lib/python3.10/site-packages',
 '/home/fr/fr_ze12/SF_hipposlam']

In [2]:
# Add the parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
print(parent_dir)
sys.path.append(parent_dir)
sys.path.remove('/home/schaffert/Documents/Hippodunk/sample-factory/.venvDMLab/src/sample-factory')

/home/schaffert/Documents/Hippodunk/sample-factory


In [9]:
# ---------------------------------------------------------------------------
# logging helpers (put near the top of the file, after imports)
# ---------------------------------------------------------------------------
import datetime, pathlib, json, pandas as pd, torch, h5py

def _ensure_parent(path: pathlib.Path):
    path.parent.mkdir(parents=True, exist_ok=True)

In [3]:
import time
from collections import deque
from typing import Dict, Tuple

import gymnasium as gym
import numpy as np
import torch
from torch import Tensor

from sample_factory.algo.learning.learner import BaseLearner, create_learner
from sample_factory.algo.sampling.batched_sampling import preprocess_actions
from sample_factory.algo.utils.action_distributions import argmax_actions
from sample_factory.algo.utils.env_info import extract_env_info
from sample_factory.algo.utils.make_env import make_env_func_batched
from sample_factory.algo.utils.misc import ExperimentStatus
from sample_factory.algo.utils.rl_utils import make_dones, prepare_and_normalize_obs
from sample_factory.algo.utils.tensor_utils import unsqueeze_tensor
from sample_factory.cfg.arguments import load_from_checkpoint
from sample_factory.huggingface.huggingface_utils import generate_model_card, generate_replay_video, push_to_hf
from sample_factory.model.actor_critic import create_actor_critic
from sample_factory.model.model_utils import get_rnn_size
from sample_factory.utils.attr_dict import AttrDict
from sample_factory.utils.typing import Config, StatusCode
from sample_factory.utils.utils import debug_log_every_n, experiment_dir, log

In [4]:
# # from sample_factory.arguments import load_from_checkpoint
# cfg_filename='../train_dir/record_distance_metric64/config.json'
# with open(cfg_filename, "r") as json_file:
#     json_params = json.load(json_file)
#     log.warning("Loading existing experiment configuration from %s", cfg_filename)
#     loaded_cfg = AttrDict(json_params)

# # # override the parameters in config file with values passed from command line
# # for key, value in cfg.cli_args.items():
# #     if key in loaded_cfg and loaded_cfg[key] != value:
# #         log.debug("Overriding arg %r with value %r passed from command line", key, value)
# #         loaded_cfg[key] = value

# # # incorporate extra CLI parameters that were not present in JSON file
# # for key, value in vars(cfg).items():
# #     if key not in loaded_cfg:
# #         log.debug("Adding new argument %r=%r that is not in the saved config file!", key, value)
# #         loaded_cfg[key] = value

In [5]:
!pwd

/home/fr/fr_ze12


In [21]:
# from sample_factory.enjoy import enjoy
from sf_working_directories.zeynep.dmlab.enjoy_hipposlam import enjoy
from sf_working_directories.zeynep.dmlab.train_hipposlam import parse_dmlab_args, register_dmlab_components

mapname="ymaze_instr"

expname='00_ymaze_norew_instrx200_modulate_see_1111'
expname_bad='01_ymaze_norew_instrx200_modulate_see_2222'
traindir_200mod = "/work/classic/fr_ze12-data/ymaze_CTRL/norew_INSTR/x200_modulate/train_dir/ymaze_norew_instrx200_modulate/ymaze_norew_instrx200_modulate_"

expname2='04_ymaze_norew_instrx9_see_5555'
expname2_bad = '03_ymaze_norew_instrx9_see_4444'
traindir_9 = '/work/classic/fr_ze12-data/ymaze_CTRL/norew_INSTR/x9/train_dir/ymaze_norew_instrx9/ymaze_norew_instrx9_'

cli = [
    "--algo", "APPO",
    "--env", mapname ,         # pick any DM‑Lab level you have
    "--experiment", expname2_bad,
    "--encoder_load_path","/home/schaffert/Documents/Hippodunk/sample-factory/train_dir/best_000025288_203030528_reward_94.185.pth",
    "--train_dir", traindir_9, # anything writable
    "--max_num_frames", "50000",          # short rollout for the test
    "--num_envs", "8",
    "--dmlab_level_cache_path","./.dmlab_cache",
    "--load_checkpoint_kind","latest",
    "--use_jit","False",
    "--with_pos_obs","True",
    "--no_render",        # <-- skip human window; avoid X11 on servers
]

cli_dict={
 'algo': 'APPO',
 'env': mapname,
 'experiment': expname2_bad,
 'encoder_load_path': '/home/schaffert/Documents/Hippodunk/sample-factory/train_dir/best_000025288_203030528_reward_94.185.pth',
 'train_dir': traindir_9,
 'max_num_frames': '50000',
 'num_envs': '8',
 'dmlab_level_cache_path': './.dmlab_cache',
 'load_checkpoint_kind': 'latest',
 'no_render': True,
 'use_jit': False,
 'with_pos_obs': True,
}
register_dmlab_components()
cfg = parse_dmlab_args(evaluation=True, argv=cli)

# tweak whatever you like *after* parsing
# cfg.with_pos_obs = True
cfg.cli_args=cli_dict
# status = enjoy(cfg)

[2026-08-18 12:02:42,705][395989] Environment ymaze already registered, overwriting...
[2026-08-18 12:02:42,706][395989] Environment ymaze_instr already registered, overwriting...
[2026-08-18 12:02:42,706][395989] Environment ymaze_noswitch already registered, overwriting...
[2026-08-18 12:02:42,707][395989] Environment openfield_map2_fixed_loc3 already registered, overwriting...
[2026-08-18 12:02:42,708][395989] Environment openfield_map2_fixed_loc1 already registered, overwriting...
[2026-08-18 12:02:42,708][395989] Environment openfield_map2_fixed_loc2 already registered, overwriting...
[2026-08-18 12:02:42,708][395989] Environment openfield_map2_fixed_loc3 already registered, overwriting...
[2026-08-18 12:02:42,709][395989] Environment openfield_map2_fixed_loc3_noreward already registered, overwriting...
[2026-08-18 12:02:42,709][395989] Environment dmlab_benchmark already registered, overwriting...
[2026-08-18 12:02:42,710][395989] Environment dmlab_30 already registered, overwrit

In [6]:
# cfg = load_from_checkpoint(cfg)

In [22]:
#### THIS is not for "enjoy", but to take the actor critic model out for other funky stuff

verbose = False

cfg = load_from_checkpoint(cfg)

eval_env_frameskip: int = cfg.env_frameskip #if cfg.eval_env_frameskip is None else cfg.eval_env_frameskip
assert (
    cfg.env_frameskip % eval_env_frameskip == 0
), f"{cfg.env_frameskip=} must be divisible by {eval_env_frameskip=}"
render_action_repeat: int = cfg.env_frameskip // eval_env_frameskip
cfg.env_frameskip = cfg.eval_env_frameskip = eval_env_frameskip
log.debug(f"Using frameskip {cfg.env_frameskip} and {render_action_repeat=} for evaluation")

cfg.num_envs = 1

render_mode = "human"
'''if cfg.save_video:
    render_mode = "rgb_array"
elif cfg.no_render:
    render_mode = None'''
render_mode = None

env = make_env_func_batched(
    cfg, env_config=AttrDict(worker_index=0, vector_index=0, env_id=0), render_mode=render_mode
)
env_info = extract_env_info(env, cfg)

if hasattr(env.unwrapped, "reset_on_init"):
    # reset call ruins the demo recording for VizDoom
    env.unwrapped.reset_on_init = False
log.info(env.action_space)
actor_critic_9_bad = create_actor_critic(cfg, env.observation_space, env.action_space)
# actor_critic.eval()

[2026-08-18 12:02:49,346][395989] Loading existing experiment configuration from /work/classic/fr_ze12-data/ymaze_CTRL/norew_INSTR/x9/train_dir/ymaze_norew_instrx9/ymaze_norew_instrx9_/03_ymaze_norew_instrx9_see_4444/config.json
[2026-08-18 12:02:49,347][395989] Overriding arg 'env' with value 'ymaze_instr' passed from command line
[2026-08-18 12:02:49,347][395989] Overriding arg 'encoder_load_path' with value '/home/schaffert/Documents/Hippodunk/sample-factory/train_dir/best_000025288_203030528_reward_94.185.pth' passed from command line
[2026-08-18 12:02:49,348][395989] Overriding arg 'train_dir' with value '/work/classic/fr_ze12-data/ymaze_CTRL/norew_INSTR/x9/train_dir/ymaze_norew_instrx9/ymaze_norew_instrx9_' passed from command line
[2026-08-18 12:02:49,349][395989] Overriding arg 'dmlab_level_cache_path' with value './.dmlab_cache' passed from command line
[2026-08-18 12:02:49,349][395989] Overriding arg 'use_jit' with value False passed from command line
[2026-08-18 12:02:49,349

In [15]:
actor_critic.eval()

ActorCriticSharedWeights(
  (obs_normalizer): ObservationNormalizer()
  (returns_normalizer): RecursiveScriptModule(original_name=RunningMeanStdInPlace)
  (encoder): HipposlamEncoder(
    (depth_encoder): DepthEncoder(
      (downsample): Upsample(size=(1, 10), mode='nearest')
    )
    (basic_encoder): ResNet18Layer2(
      (features): Sequential(
        (0): Sequential(
          (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
          (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (2): ReLU(inplace=True)
          (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
        )
        (1): Sequential(
          (0): BasicBlock(
            (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
            (relu): ReLU(inplac

In [15]:
for name, param in actor_critic.named_parameters():
    if "decoder" in name:
        print(param.shape)


torch.Size([128, 63])
torch.Size([128])
torch.Size([128, 128])
torch.Size([128])


In [13]:
actor_critic_200mod.encoder.instruction_embed_layer.weight

Parameter containing:
tensor([[ 0.1615, -0.0325, -0.1558],
        [-0.0624, -0.2191,  0.1604],
        [ 0.2920, -0.1186, -0.0990],
        [ 0.2665,  0.0187, -0.2932],
        [-0.0379, -0.2145,  0.2591],
        [ 0.0913,  0.4579,  0.0136],
        [ 0.1947,  0.2448, -0.0734],
        [-0.3081,  0.2287, -0.0417],
        [ 0.3506,  0.3515,  0.1916],
        [-0.2035, -0.1373,  0.1074],
        [-0.2230,  0.3360, -0.3969],
        [-0.2185,  0.0102, -0.2967],
        [ 0.2235,  0.3941,  0.1938],
        [ 0.0666, -0.2709,  0.0959],
        [-0.1102,  0.1894,  0.6615],
        [-0.5855,  0.2204,  0.0396]], requires_grad=True)

In [32]:
actor_critic_200mod.encoder.DG_projection.linear.weight

Parameter containing:
tensor([[ 0.0061, -0.0071, -0.0164,  ...,  0.0092,  0.0104,  0.0137],
        [-0.0144,  0.0046, -0.0023,  ..., -0.0077, -0.0059, -0.0063],
        [-0.0003,  0.0148, -0.0144,  ...,  0.0390,  0.0352,  0.0108],
        ...,
        [ 0.0066, -0.0070, -0.0034,  ...,  0.0020, -0.0009, -0.0027],
        [ 0.0115, -0.0123, -0.0184,  ...,  0.0015, -0.0155, -0.0088],
        [-0.0257,  0.0144,  0.0076,  ...,  0.0011, -0.0160, -0.0021]],
       requires_grad=True)

In [6]:
actor_critic_200mod_bad.encoder.instruction_embed_layer.weight

Parameter containing:
tensor([[ 0.0558, -0.0463, -0.2677],
        [-0.4275,  0.1075, -0.4245],
        [-0.3877, -0.2773, -0.0311],
        [-0.0279,  0.0937, -0.1844],
        [ 0.1155, -0.1516,  0.2289],
        [ 0.0017, -0.0536,  0.3737],
        [ 0.0631,  0.0425, -0.0600],
        [-0.1709,  0.0062,  0.3975],
        [-0.6645,  0.1351,  0.1884],
        [ 0.1539,  0.2728, -0.2169],
        [ 0.0241,  0.6888,  0.0360],
        [ 0.1248, -0.2848, -0.2885],
        [ 0.2407, -0.1500,  0.1436],
        [ 0.0118, -0.0937,  0.3701],
        [ 0.1718,  0.4366,  0.1746],
        [-0.2180,  0.0529,  0.0654]], requires_grad=True)

In [35]:
actor_critic_200mod_bad.encoder.DG_projection.linear.weight

Parameter containing:
tensor([[-0.0074,  0.0249, -0.0058,  ...,  0.0174, -0.0052, -0.0194],
        [ 0.0155,  0.0049, -0.0093,  ..., -0.0053,  0.0053,  0.0033],
        [ 0.0153, -0.0063,  0.0308,  ...,  0.0117, -0.0222,  0.0291],
        ...,
        [ 0.0025, -0.0164, -0.0132,  ..., -0.0181, -0.0213, -0.0035],
        [ 0.0074,  0.0004, -0.0181,  ...,  0.0021, -0.0186,  0.0123],
        [ 0.0160,  0.0035, -0.0139,  ...,  0.0045,  0.0049,  0.0010]],
       requires_grad=True)

In [ ]:
actor_critic.encoder.DG

HipposlamEncoder(
  (depth_encoder): DepthEncoder(
    (downsample): Upsample(size=(1, 10), mode='nearest')
  )
  (basic_encoder): ResnetEncoder(
    (conv_head): Sequential(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (2): ResBlock(
        (res_block_core): Sequential(
          (0): ReLU()
          (1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (2): ReLU()
          (3): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        )
      )
      (3): ResBlock(
        (res_block_core): Sequential(
          (0): ReLU()
          (1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (2): ReLU()
          (3): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        )
      )
      (4): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (5): MaxPool2d

In [23]:
## Single enjoy run


verbose = False

#cfg = load_from_checkpoint(cfg)

eval_env_frameskip: int = cfg.env_frameskip 
assert (
    cfg.env_frameskip % eval_env_frameskip == 0
), f"{cfg.env_frameskip=} must be divisible by {eval_env_frameskip=}"
render_action_repeat: int = cfg.env_frameskip // eval_env_frameskip
cfg.env_frameskip = cfg.eval_env_frameskip = eval_env_frameskip
log.debug(f"Using frameskip {cfg.env_frameskip} and {render_action_repeat=} for evaluation")

cfg.num_envs = 1

# render_mode = "human"
# if cfg.save_video:
#     render_mode = "rgb_array"
# elif cfg.no_render:
render_mode = None

env = make_env_func_batched(
    cfg, env_config=AttrDict(worker_index=0, vector_index=0, env_id=0), render_mode=render_mode
)
env_info = extract_env_info(env, cfg)

if hasattr(env.unwrapped, "reset_on_init"):
    # reset call ruins the demo recording for VizDoom
    env.unwrapped.reset_on_init = False
log.info(env.action_space)
actor_critic = create_actor_critic(cfg, env.observation_space, env.action_space)
actor_critic.eval()



device = torch.device("cpu" if cfg.device == "cpu" else "cuda")
actor_critic.model_to_device(device)

# learner = create_learner(cfg,env_info,)


#################### register hook
layers_to_log = [
    # 'encoder.basic_encoder.mlp_layers.0',
    "encoder.DG_projection.linear",
#    "encoder.instruction_embed_layer",
    "core",
    
    # "decoder.mlp.0",
    # "decoder.mlp.2"
]          # <- example; edit to taste

# 2B. activation buffer

import collections
act_buffers = collections.defaultdict(list)

def make_hook(layer_name):
    def _hook(_m, _inp, out):
        if isinstance(out, (tuple, list)):      # RNN returns (output, h_n)
            out = out[0]
        act_buffers[layer_name].append(out.detach().cpu())
    return _hook

# attach the hook once
for layer_to_log in layers_to_log:
    try:
        dict(actor_critic.named_modules())[layer_to_log].register_forward_hook(make_hook(layer_to_log))
        log.info("Activation hook registered on %s", layer_to_log)
    except KeyError:
        raise RuntimeError(f"Layer '{layer_to_log}' not found in the network!")

####################


policy_id = cfg.policy_index
# log.info(policy_id)
name_prefix = dict(latest="checkpoint", best="best")[cfg.load_checkpoint_kind]
# log.info(Learner.checkpoint_dir(cfg, policy_id))
checkpoints = BaseLearner.get_checkpoints(BaseLearner.checkpoint_dir(cfg, policy_id), f"{name_prefix}_*")
checkpoint_dict = BaseLearner.load_checkpoint(checkpoints, device)
actor_critic.load_state_dict(checkpoint_dict["model"])

episode_rewards = [deque([], maxlen=100) for _ in range(env.num_agents)]
true_objectives = [deque([], maxlen=100) for _ in range(env.num_agents)]
num_frames = 0

last_render_start = time.time()

def max_frames_reached(frames):
    return cfg.max_num_frames is not None and frames > cfg.max_num_frames

reward_list = []

obs, infos = env.reset()
rnn_states = torch.zeros([env.num_agents, get_rnn_size(cfg)], dtype=torch.float32, device=device)
episode_reward = None
finished_episode = [False for _ in range(env.num_agents)]

video_frames = []
num_episodes = 0
num_traj = 0

# saved_data=dict()
pose_records = []

with torch.no_grad():
    while not max_frames_reached(num_frames):
        normalized_obs = prepare_and_normalize_obs(actor_critic, obs)

        # if not cfg.no_render:
        #     visualize_policy_inputs(normalized_obs)
        policy_outputs = actor_critic(normalized_obs, rnn_states)

        # sample actions from the distribution by default
        actions = policy_outputs["actions"]

        if cfg.eval_deterministic:
            action_distribution = actor_critic.action_distribution()
            actions = argmax_actions(action_distribution)

        # actions shape should be [num_agents, num_actions] even if it's [1, 1]
        if actions.ndim == 1:
            actions = unsqueeze_tensor(actions, dim=-1)
        actions = preprocess_actions(env_info, actions)

        rnn_states = policy_outputs["new_rnn_states"]

        for _ in range(render_action_repeat):
            # last_render_start = render_frame(cfg, env, video_frames, num_episodes, last_render_start)

            obs, rew, terminated, truncated, infos = env.step(actions)
            # log.info(obs['DEBUG.POS.TRANS'])
            # log.info(terminated)
            # save info
            frame_idx = num_frames          # or use a wall‑clock timestamp
            pos = obs['DEBUG.POS.TRANS']    # (B,3)
            rot = obs['DEBUG.POS.ROT']      # (B,3) or (B,4) depending on env

            for agent_i in range(env.num_agents):
                pose_records.append({
                    "frame"     : frame_idx,
                    "agent"     : agent_i,
                    "x"         : float(pos[agent_i, 0]),
                    "y"         : float(pos[agent_i, 1]),
                    "z"         : float(pos[agent_i, 2]),
                    "rot_x"     : float(rot[agent_i, 0]),
                    "rot_y"     : float(rot[agent_i, 1]),
                    "rot_z"     : float(rot[agent_i, 2]),
                    "num_traj"  : num_traj,
                    # keep the whole info dict as a JSON string for convenience
                    "info"      : json.dumps(infos[agent_i], default=str),
                })



            dones = make_dones(terminated, truncated)
            # log.info(dones)
            infos = [{} for _ in range(env_info.num_agents)] if infos is None else infos

            if episode_reward is None:
                episode_reward = rew.float().clone()
            else:
                episode_reward += rew.float()

            num_frames += 1
            if num_frames % 100 == 0:
                log.debug(f"Num frames {num_frames}...")

            dones = dones.cpu().numpy()
            for agent_i, done_flag in enumerate(dones):
                if done_flag:
                    num_traj += 1
                    log.info(done_flag)
                    log.info(cfg.use_record_episode_statistics)
                    finished_episode[agent_i] = True
                    rew = episode_reward[agent_i].item()
                    episode_rewards[agent_i].append(rew)

                    true_objective = rew
                    if isinstance(infos, (list, tuple)):
                        true_objective = infos[agent_i].get("true_objective", rew)
                    true_objectives[agent_i].append(true_objective)

                    if verbose:
                        log.info(
                            "Episode finished for agent %d at %d frames. Reward: %.3f, true_objective: %.3f",
                            agent_i,
                            num_frames,
                            episode_reward[agent_i],
                            true_objectives[agent_i][-1],
                        )
                    rnn_states[agent_i] = torch.zeros([get_rnn_size(cfg)], dtype=torch.float32, device=device)
                    episode_reward[agent_i] = 0

                    if cfg.use_record_episode_statistics:
                        # we want the scores from the full episode not a single agent death (due to EpisodicLifeEnv wrapper)
                        if "episode" in infos[agent_i].keys():
                            num_episodes += 1
                            reward_list.append(infos[agent_i]["episode"]["r"])
                    else:
                        num_episodes += 1
                        reward_list.append(true_objective)

            # if episode terminated synchronously for all agents, pause a bit before starting a new one
            if all(dones):
                # render_frame(cfg, env, video_frames, num_episodes, last_render_start)
                time.sleep(0.05)

            if all(finished_episode):
                finished_episode = [False] * env.num_agents
                avg_episode_rewards_str, avg_true_objective_str = "", ""
                for agent_i in range(env.num_agents):
                    avg_rew = np.mean(episode_rewards[agent_i])
                    avg_true_obj = np.mean(true_objectives[agent_i])

                    if not np.isnan(avg_rew):
                        if avg_episode_rewards_str:
                            avg_episode_rewards_str += ", "
                        avg_episode_rewards_str += f"#{agent_i}: {avg_rew:.3f}"
                    if not np.isnan(avg_true_obj):
                        if avg_true_objective_str:
                            avg_true_objective_str += ", "
                        avg_true_objective_str += f"#{agent_i}: {avg_true_obj:.3f}"

                log.info(
                    "Avg episode rewards: %s, true rewards: %s", avg_episode_rewards_str, avg_true_objective_str
                )
                log.info(
                    "Avg episode reward: %.3f, avg true_objective: %.3f",
                    np.mean([np.mean(episode_rewards[i]) for i in range(env.num_agents)]),
                    np.mean([np.mean(true_objectives[i]) for i in range(env.num_agents)]),
                )

            # VizDoom multiplayer stuff
            # for player in [1, 2, 3, 4, 5, 6, 7, 8]:
            #     key = f'PLAYER{player}_FRAGCOUNT'
            #     if key in infos[0]:
            #         log.debug('Score for player %d: %r', player, infos[0][key])
        # log.info(num_episodes)
        if num_episodes >= cfg.max_num_episodes:
            break

env.close()

[2026-08-18 12:02:59,295][395989] Using frameskip 8 and render_action_repeat=1 for evaluation
[2026-08-18 12:02:59,296][395989] Using GPU 0 for DMLab rendering!
[2026-08-18 12:02:59,297][395989] {'worker_index': 0, 'vector_index': 0, 'env_id': 0} level ymaze_vol3_INSTR task id 0
[2026-08-18 12:02:59,298][395989] True
[2026-08-18 12:02:59,298][395989] True
[2026-08-18 12:02:59,299][395989] depth_sensor  dmlab env True
[2026-08-18 12:02:59,299][395989] REWARD INPUT IS DISABLED
[2026-08-18 12:03:00,842][395989] using reduced action set!
[2026-08-18 12:03:00,850][395989] Discrete(5)
[2026-08-18 12:03:00,851][395989] RunningMeanStd input shape: (1,)
[2026-08-18 12:03:00,864][395989] True
[2026-08-18 12:03:00,865][395989] using depth sensor True
[2026-08-18 12:03:00,866][395989] Num input channels for depth encoder: 1
[2026-08-18 12:03:00,867][395989] original obs space: Box(0, 255, (4, 72, 96), uint8)
[2026-08-18 12:03:00,985][395989] DMLab policy head output size: 3843
[2026-08-18 12:03:00

Entered LEFT reward zone!
Entered LEFT reward zone!
Entered LEFT reward zone!
Entered LEFT reward zone!
Entered LEFT reward zone!
Entered LEFT reward zone!
Entered LEFT reward zone!
Entered LEFT reward zone!
Reward predicted LEFT
Entered LEFT reward zone!
Entered LEFT reward zone!
--- END OF BLOCK 1 (15 Trials Completed) ---
[Shift Roll: 0.58 <= 0.66 -> [CONTINGENCY INVERSION FOR BLOCK 2]]
Entered RIGHT reward zone!
Entered RIGHT reward zone!
Entered RIGHT reward zone!
Entered RIGHT reward zone!
Entered RIGHT reward zone!
Entered RIGHT reward zone!
Reward predicted RIGHT
Entered RIGHT reward zone!
Entered RIGHT reward zone!
Entered RIGHT reward zone!
Entered RIGHT reward zone!
Reward predicted RIGHT
Entered RIGHT reward zone!
--- END OF BLOCK 2 (15 Trials Completed) ---
[Shift Roll: 0.28 <= 0.66 -> [CONTINGENCY INVERSION FOR BLOCK 3]]
Entered LEFT reward zone!
Reward predicted LEFT
Entered LEFT reward zone!
Reward predicted LEFT
Entered LEFT reward zone!
Entered LEFT reward zone!
Rewar

[2026-08-18 12:03:02,006][395989] Num frames 100...
[2026-08-18 12:03:02,502][395989] Num frames 200...
[2026-08-18 12:03:02,995][395989] Num frames 300...
[2026-08-18 12:03:03,486][395989] Num frames 400...
[2026-08-18 12:03:03,978][395989] Num frames 500...
[2026-08-18 12:03:04,471][395989] Num frames 600...
[2026-08-18 12:03:04,962][395989] Num frames 700...
[2026-08-18 12:03:05,457][395989] Num frames 800...
[2026-08-18 12:03:05,952][395989] Num frames 900...
[2026-08-18 12:03:06,452][395989] Num frames 1000...
[2026-08-18 12:03:06,951][395989] Num frames 1100...
[2026-08-18 12:03:07,444][395989] Num frames 1200...
[2026-08-18 12:03:07,936][395989] Num frames 1300...
[2026-08-18 12:03:08,429][395989] Num frames 1400...
[2026-08-18 12:03:08,928][395989] Num frames 1500...
[2026-08-18 12:03:09,425][395989] Num frames 1600...
[2026-08-18 12:03:09,920][395989] Num frames 1700...
[2026-08-18 12:03:10,403][395989] Num frames 1800...
[2026-08-18 12:03:10,904][395989] Num frames 1900...
[2

--- END OF BLOCK 1 (15 Trials Completed) ---
[Shift Roll: 0.46 <= 0.66 -> [CONTINGENCY INVERSION FOR BLOCK 2]]
Entered LEFT reward zone!
Reward predicted LEFT
--- END OF BLOCK 2 (15 Trials Completed) ---
[Shift Roll: 0.76 > 0.66 -> [CONTINGENCIES REMAIN THE SAME FOR BLOCK 3]]
Entered LEFT reward zone!
Reward predicted LEFT
--- END OF BLOCK 3 (15 Trials Completed) ---
[Shift Roll: 0.53 <= 0.66 -> [CONTINGENCY INVERSION FOR BLOCK 4]]
--- END OF BLOCK 4 (15 Trials Completed) ---
[Shift Roll: 0.15 <= 0.66 -> [CONTINGENCY INVERSION FOR BLOCK 5]]
--- END OF BLOCK 5 (15 Trials Completed) ---
[Shift Roll: 0.65 <= 0.66 -> [CONTINGENCY INVERSION FOR BLOCK 6]]
--- 5 BLOCKS COMPLETED ---

--- BLOCK 1 STARTED ---


[2026-08-18 12:03:19,820][395989] Num frames 3600...
[2026-08-18 12:03:20,298][395989] Num frames 3700...
[2026-08-18 12:03:20,780][395989] Num frames 3800...
[2026-08-18 12:03:21,270][395989] Num frames 3900...
[2026-08-18 12:03:21,749][395989] Num frames 4000...
[2026-08-18 12:03:22,233][395989] Num frames 4100...
[2026-08-18 12:03:22,704][395989] Num frames 4200...
[2026-08-18 12:03:23,183][395989] Num frames 4300...
[2026-08-18 12:03:23,671][395989] Num frames 4400...
[2026-08-18 12:03:24,312][395989] Num frames 4500...
[2026-08-18 12:03:24,787][395989] Num frames 4600...
[2026-08-18 12:03:25,264][395989] Num frames 4700...
[2026-08-18 12:03:25,731][395989] Num frames 4800...
[2026-08-18 12:03:26,216][395989] Num frames 4900...
[2026-08-18 12:03:26,709][395989] Num frames 5000...
[2026-08-18 12:03:27,193][395989] Num frames 5100...
[2026-08-18 12:03:27,672][395989] Num frames 5200...
[2026-08-18 12:03:28,154][395989] Num frames 5300...
[2026-08-18 12:03:28,646][395989] Num frames 5

...

[2026-08-18 12:06:43,017][395989] Num frames 42800...
[2026-08-18 12:06:43,462][395989] Num frames 42900...
[2026-08-18 12:06:43,900][395989] Num frames 43000...
[2026-08-18 12:06:44,330][395989] Num frames 43100...
[2026-08-18 12:06:44,779][395989] Num frames 43200...
[2026-08-18 12:06:45,221][395989] Num frames 43300...
[2026-08-18 12:06:45,708][395989] Num frames 43400...
[2026-08-18 12:06:46,192][395989] Num frames 43500...
[2026-08-18 12:06:46,679][395989] Num frames 43600...
[2026-08-18 12:06:47,165][395989] Num frames 43700...
[2026-08-18 12:06:47,650][395989] Num frames 43800...
[2026-08-18 12:06:48,145][395989] Num frames 43900...
[2026-08-18 12:06:48,643][395989] Num frames 44000...
[2026-08-18 12:06:49,127][395989] Num frames 44100...
[2026-08-18 12:06:49,624][395989] Num frames 44200...
[2026-08-18 12:06:50,123][395989] Num frames 44300...
[2026-08-18 12:06:50,625][395989] Num frames 44400...
[2026-08-18 12:06:51,128][395989] Num frames 44500...
[2026-08-18 12:06:51,628][39

Entered LEFT reward zone!
--- END OF BLOCK 1 (15 Trials Completed) ---
[Shift Roll: 0.78 > 0.66 -> [CONTINGENCIES REMAIN THE SAME FOR BLOCK 2]]
--- END OF BLOCK 2 (15 Trials Completed) ---
[Shift Roll: 0.66 <= 0.66 -> [CONTINGENCY INVERSION FOR BLOCK 3]]
--- END OF BLOCK 3 (15 Trials Completed) ---
[Shift Roll: 0.78 > 0.66 -> [CONTINGENCIES REMAIN THE SAME FOR BLOCK 4]]
Entered LEFT reward zone!
Reward predicted LEFT
--- END OF BLOCK 4 (15 Trials Completed) ---
[Shift Roll: 0.14 <= 0.66 -> [CONTINGENCY INVERSION FOR BLOCK 5]]
--- END OF BLOCK 5 (15 Trials Completed) ---
[Shift Roll: 0.43 <= 0.66 -> [CONTINGENCY INVERSION FOR BLOCK 6]]
--- 5 BLOCKS COMPLETED ---

--- BLOCK 1 STARTED ---


[2026-08-18 12:07:00,655][395989] Num frames 46300...
[2026-08-18 12:07:01,137][395989] Num frames 46400...
[2026-08-18 12:07:01,623][395989] Num frames 46500...
[2026-08-18 12:07:02,108][395989] Num frames 46600...
[2026-08-18 12:07:02,598][395989] Num frames 46700...
[2026-08-18 12:07:03,098][395989] Num frames 46800...
[2026-08-18 12:07:03,598][395989] Num frames 46900...
[2026-08-18 12:07:04,101][395989] Num frames 47000...
[2026-08-18 12:07:04,600][395989] Num frames 47100...
[2026-08-18 12:07:05,100][395989] Num frames 47200...
[2026-08-18 12:07:05,600][395989] Num frames 47300...
[2026-08-18 12:07:06,100][395989] Num frames 47400...
[2026-08-18 12:07:06,599][395989] Num frames 47500...
[2026-08-18 12:07:07,105][395989] Num frames 47600...
[2026-08-18 12:07:07,607][395989] Num frames 47700...
[2026-08-18 12:07:08,108][395989] Num frames 47800...
[2026-08-18 12:07:08,612][395989] Num frames 47900...
[2026-08-18 12:07:09,115][395989] Num frames 48000...
[2026-08-18 12:07:09,620][39

--- END OF BLOCK 1 (15 Trials Completed) ---
[Shift Roll: 0.57 <= 0.66 -> [CONTINGENCY INVERSION FOR BLOCK 2]]
Entered LEFT reward zone!
Reward predicted LEFT
--- END OF BLOCK 2 (15 Trials Completed) ---
[Shift Roll: 0.77 > 0.66 -> [CONTINGENCIES REMAIN THE SAME FOR BLOCK 3]]
--- END OF BLOCK 3 (15 Trials Completed) ---
[Shift Roll: 0.06 <= 0.66 -> [CONTINGENCY INVERSION FOR BLOCK 4]]
--- END OF BLOCK 4 (15 Trials Completed) ---
[Shift Roll: 0.91 > 0.66 -> [CONTINGENCIES REMAIN THE SAME FOR BLOCK 5]]
--- END OF BLOCK 5 (15 Trials Completed) ---
[Shift Roll: 0.24 <= 0.66 -> [CONTINGENCY INVERSION FOR BLOCK 6]]
--- 5 BLOCKS COMPLETED ---

--- BLOCK 1 STARTED ---


[2026-08-18 12:07:19,232][395989] Num frames 49900...
[2026-08-18 12:07:19,710][395989] Num frames 50000...


In [52]:
pose_records[0]['info']

'{"num_frames": 8, "highrew_hit": false, "highrew_miss": false, "lowrew_hit": false, "lowrew_miss": false}'

In [24]:
ts        = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
telemetry = pathlib.Path(experiment_dir(cfg=cfg)) / "telemetry"
_ensure_parent(telemetry)
pose_path = telemetry / f"pose_{ts}.parquet"

csv_path = pose_path.with_suffix(".csv")
_ensure_parent(csv_path)
pddata=pd.DataFrame(pose_records)
pddata.to_csv(csv_path, index=False)
log.info("Saved %d pose rows to %s", len(pose_records), csv_path)

# log.info("Saved %d pose rows to %s", len(pose_records), pose_path)

# 5B.  save activations to HDF5  ----------------------------------------
act_path = telemetry / f"activations_{ts}.h5"
with h5py.File(act_path, "w") as h5:
    for layer, lst in act_buffers.items():
        if not lst:            # nothing recorded for that layer
            continue
        data = torch.cat(lst, dim=0).numpy()   # (frames*agents, …)
        h5.create_dataset(layer, data=data, compression="gzip")
        log.info("Saved %-20s  shape=%r", layer, data.shape)

[2026-08-18 12:07:32,240][395989] Saved 50001 pose rows to /work/classic/fr_ze12-data/ymaze_CTRL/norew_INSTR/x9/train_dir/ymaze_norew_instrx9/ymaze_norew_instrx9_/03_ymaze_norew_instrx9_see_4444/telemetry/pose_20260818_120731.csv
[2026-08-18 12:07:32,394][395989] Saved encoder.DG_projection.linear  shape=(50001, 16)
[2026-08-18 12:07:33,157][395989] Saved core                  shape=(50001, 1149)
